In [1]:
import torch

print(torch.cuda.is_available())

True


In [2]:
!pip install -q transformers torch torchaudio scikit-learn soundfile

In [3]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

MODEL_NAME = "Vansh180/deepfake-audio-wav2vec2"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME)

model.eval()

print("Model loaded successfully!")
print(model.config.id2label)

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Model loaded successfully!
{0: 'real', 1: 'fake'}


In [4]:
!pip install -q datasets soundfile librosa scikit-learn

In [6]:
from datasets import load_dataset

ds = load_dataset(
    "SpeechAntiSpoofingBenchmarks/ASVspoof2021_DF",
    split="test",
    streaming=True
)

print(ds)

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

IterableDataset({
    features: ['path', 'audio', 'label', 'notes'],
    num_shards: 80
})


In [7]:
sample = next(iter(ds))

print("Keys:", sample.keys())
print("Path:", sample["path"])
print("Label:", sample["label"])
print("Notes:", sample["notes"])

print("\nAudio information:")
print(sample["audio"])

Keys: dict_keys(['path', 'audio', 'label', 'notes'])
Path: DF_E_2000011.flac
Label: 1
Notes: {"utterance_id": "DF_E_2000011", "speaker_id": "LA_0023", "subset": "progress", "codec": "nocodec", "source": "asvspoof", "attack_id": "A14", "vocoder": "traditional_vocoder"}

Audio information:


In [8]:
print("Label names:", ds.features["label"])

Label names: ClassLabel(names=['bonafide', 'spoof'])


In [9]:
sample = next(iter(ds))

audio_data = sample["audio"].get_all_samples()

waveform = audio_data.data
sample_rate = audio_data.sample_rate

print("Waveform shape:", waveform.shape)
print("Sample rate:", sample_rate)
print("Label:", sample["label"])

Waveform shape: torch.Size([1, 22801])
Sample rate: 16000
Label: 1


In [10]:
import torch

# Get waveform
audio = waveform.squeeze().numpy()

# Send audio to Wav2Vec2 feature extractor
inputs = feature_extractor(
    audio,
    sampling_rate=sample_rate,
    return_tensors="pt"
)

# Run the model
with torch.no_grad():
    outputs = model(**inputs)

# Convert model output into probabilities
probabilities = torch.softmax(outputs.logits, dim=-1)[0]

real_probability = probabilities[0].item()
fake_probability = probabilities[1].item()

if fake_probability >= real_probability:
    prediction = "AI_GENERATED"
else:
    prediction = "GENUINE"

print("Prediction:", prediction)
print("Real probability:", real_probability)
print("Fake probability:", fake_probability)
print("Actual dataset label:", "SPOOF" if sample["label"] == 1 else "BONAFIDE")

Prediction: GENUINE
Real probability: 0.8537881970405579
Fake probability: 0.14621184766292572
Actual dataset label: SPOOF


In [11]:
import numpy as np
import pandas as pd
import torch

# Fresh dataset iterator
iterator = iter(ds)

real_samples = []
fake_samples = []

# Collect 10 real + 10 fake samples
while len(real_samples) < 10 or len(fake_samples) < 10:

    sample = next(iterator)

    if sample["label"] == 0 and len(real_samples) < 10:
        real_samples.append(sample)

    elif sample["label"] == 1 and len(fake_samples) < 10:
        fake_samples.append(sample)

print("Real samples:", len(real_samples))
print("Fake samples:", len(fake_samples))

Real samples: 10
Fake samples: 10


In [12]:
import pandas as pd
import torch

results = []

all_samples = real_samples + fake_samples

for sample in all_samples:

    # Get audio
    audio_data = sample["audio"].get_all_samples()

    waveform = audio_data.data
    sample_rate = audio_data.sample_rate

    # Convert to numpy
    audio = waveform.squeeze().numpy()

    # Give audio to model
    inputs = feature_extractor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    # Model prediction
    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    real_probability = probabilities[0].item()
    fake_probability = probabilities[1].item()

    # Model decision
    if fake_probability >= real_probability:
        prediction = "FAKE"
    else:
        prediction = "REAL"

    # Actual label
    actual = "FAKE" if sample["label"] == 1 else "REAL"

    results.append({
        "file": sample["path"],
        "actual": actual,
        "prediction": prediction,
        "real_probability": real_probability,
        "fake_probability": fake_probability
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

             file actual prediction  real_probability  fake_probability
DF_E_2000053.flac   REAL       REAL          0.937972          0.062028
DF_E_2000058.flac   REAL       REAL          0.936016          0.063984
DF_E_2000079.flac   REAL       REAL          0.938139          0.061861
DF_E_2000246.flac   REAL       REAL          0.919709          0.080291
DF_E_2000503.flac   REAL       REAL          0.928014          0.071986
DF_E_2000508.flac   REAL       REAL          0.929693          0.070307
DF_E_2000531.flac   REAL       REAL          0.914033          0.085967
DF_E_2000676.flac   REAL       REAL          0.917519          0.082481
DF_E_2001149.flac   REAL       REAL          0.930106          0.069894
DF_E_2001163.flac   REAL       REAL          0.935589          0.064411
DF_E_2000011.flac   FAKE       REAL          0.853788          0.146212
DF_E_2000013.flac   FAKE       REAL          0.924812          0.075188
DF_E_2000024.flac   FAKE       REAL          0.904054          0

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true = results_df["actual"]
y_pred = results_df["prediction"]

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    pos_label="FAKE",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label="FAKE",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label="FAKE",
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["REAL", "FAKE"]
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["REAL", "FAKE"],
        zero_division=0
    )
)

Accuracy : 0.5
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0

Confusion Matrix:
[[10  0]
 [10  0]]

Classification Report:
              precision    recall  f1-score   support

        REAL       0.50      1.00      0.67        10
        FAKE       0.00      0.00      0.00        10

    accuracy                           0.50        20
   macro avg       0.25      0.50      0.33        20
weighted avg       0.25      0.50      0.33        20



In [14]:
# Check the first 1000 samples from the ASVspoof DF test set

from collections import Counter

iterator = iter(ds)

labels = []
subsets = []
codecs = []
attacks = []

for i in range(1000):
    sample = next(iterator)

    labels.append(sample["label"])
    subsets.append(sample["notes"].get("subset"))
    codecs.append(sample["notes"].get("codec"))
    attacks.append(sample["notes"].get("attack_id"))

print("Total samples checked:", len(labels))

print("\nLabel distribution:")
print(Counter(labels))

print("\nSubset distribution:")
print(Counter(subsets))

print("\nCodec distribution:")
print(Counter(codecs))

print("\nNumber of different attack IDs:")
print(len(set(attacks)))

AttributeError: 'str' object has no attribute 'get'

In [15]:
from collections import Counter
import json

iterator = iter(ds)

labels = []
subsets = []
codecs = []
attacks = []

for i in range(1000):
    sample = next(iterator)

    labels.append(sample["label"])

    # Convert notes string into dictionary
    notes = json.loads(sample["notes"])

    subsets.append(notes.get("subset"))
    codecs.append(notes.get("codec"))
    attacks.append(notes.get("attack_id"))

print("Total samples checked:", len(labels))

print("\nLabel distribution:")
print(Counter(labels))

print("\nSubset distribution:")
print(Counter(subsets))

print("\nCodec distribution:")
print(Counter(codecs))

print("\nNumber of different attack IDs:")
print(len(set(attacks)))

Total samples checked: 1000

Label distribution:
Counter({1: 962, 0: 38})

Subset distribution:
Counter({'eval': 880, 'progress': 97, 'hidden': 23})

Codec distribution:
Counter({'low_m4a': 129, 'nocodec': 122, 'low_ogg': 116, 'mp3m4a': 114, 'high_m4a': 108, 'high_ogg': 108, 'oggm4a': 107, 'high_mp3': 104, 'low_mp3': 92})

Number of different attack IDs:
111


In [16]:
import json
import random

# Shuffle the streaming dataset
shuffled_ds = ds.shuffle(
    seed=42,
    buffer_size=10000
)

iterator = iter(shuffled_ds)

real_samples = []
fake_samples = []

while len(real_samples) < 100 or len(fake_samples) < 100:

    sample = next(iterator)

    if sample["label"] == 0 and len(real_samples) < 100:
        real_samples.append(sample)

    elif sample["label"] == 1 and len(fake_samples) < 100:
        fake_samples.append(sample)

print("REAL samples:", len(real_samples))
print("FAKE samples:", len(fake_samples))

REAL samples: 100
FAKE samples: 100


In [17]:
import pandas as pd
import torch

results_200 = []

all_samples_200 = real_samples + fake_samples

for i, sample in enumerate(all_samples_200):

    # Get audio
    audio_data = sample["audio"].get_all_samples()

    waveform = audio_data.data
    sample_rate = audio_data.sample_rate

    # Convert to numpy
    audio = waveform.squeeze().numpy()

    # Prepare audio for Wav2Vec2
    inputs = feature_extractor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    )

    # Model prediction
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits to probabilities
    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    real_probability = probabilities[0].item()
    fake_probability = probabilities[1].item()

    # Prediction
    if fake_probability >= real_probability:
        prediction = "FAKE"
    else:
        prediction = "REAL"

    # Actual label
    actual = "FAKE" if sample["label"] == 1 else "REAL"

    results_200.append({
        "file": sample["path"],
        "actual": actual,
        "prediction": prediction,
        "real_probability": real_probability,
        "fake_probability": fake_probability
    })

    # Progress
    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/200")

results_200_df = pd.DataFrame(results_200)

print("\nTesting completed!")
print(results_200_df.head())

Processed 20/200
Processed 40/200
Processed 60/200
Processed 80/200
Processed 100/200
Processed 120/200
Processed 140/200
Processed 160/200
Processed 180/200
Processed 200/200

Testing completed!
                file actual prediction  real_probability  fake_probability
0  DF_E_3272211.flac   REAL       REAL          0.938020          0.061980
1  DF_E_3271183.flac   REAL       REAL          0.881889          0.118111
2  DF_E_3261154.flac   REAL       REAL          0.926190          0.073810
3  DF_E_3273291.flac   REAL       REAL          0.901381          0.098619
4  DF_E_3258941.flac   REAL       REAL          0.934249          0.065751


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true = results_200_df["actual"]
y_pred = results_200_df["prediction"]

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    pos_label="FAKE",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label="FAKE",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label="FAKE",
    zero_division=0
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["REAL", "FAKE"]
)

print("===== Wav2Vec2 Benchmark Results =====")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["REAL", "FAKE"],
        zero_division=0
    )
)

===== Wav2Vec2 Benchmark Results =====
Accuracy : 0.5000
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000

Confusion Matrix:
[[100   0]
 [100   0]]

Classification Report:
              precision    recall  f1-score   support

        REAL       0.50      1.00      0.67       100
        FAKE       0.00      0.00      0.00       100

    accuracy                           0.50       200
   macro avg       0.25      0.50      0.33       200
weighted avg       0.25      0.50      0.33       200



In [19]:
!pip install -q huggingface_hub

In [22]:
from huggingface_hub import list_models

models = list(list_models(
    search="RawNet2 deepfake audio",
    limit=10
))

for m in models:
    print(m.id)

In [23]:
from huggingface_hub import list_models

models = list(list_models(search="RawNet2", limit=10))

print("Number of models found:", len(models))

for m in models:
    print(m.id)

Number of models found: 10
MattyB95/pre_trained_DF_RawNet2
rpeddu/enhanced-rawnet2-antispoofing
PatricioMN/rawnet2-3s
ALLA1N/rawnet2-la-clean-specialist
ALLA1N/rawnet2-itw-robustness-specialist
rmachado23/rawnet2
caa-speech-detection-asvspoof2019/rawnet2
caa-speech-detection-asvspoof2019/rawnet2-plateau
Arya12367/voicefence-rawnet2-codecaug
Arya12367/voicefence-rawnet2-v3-hien


In [24]:
from huggingface_hub import model_info

info = model_info("MattyB95/pre_trained_DF_RawNet2")

print("Model:", info.id)
print("Downloads:", info.downloads)
print("Tags:", info.tags)

Model: MattyB95/pre_trained_DF_RawNet2
Downloads: 0
Tags: ['license:mit', 'region:us']


In [25]:
from huggingface_hub import model_info

info = model_info("caa-speech-detection-asvspoof2019/rawnet2")

print("Model:", info.id)
print("Downloads:", info.downloads)
print("Tags:", info.tags)

Model: caa-speech-detection-asvspoof2019/rawnet2
Downloads: 0
Tags: ['pytorch', 'tensorboard', 'audio-classification', 'anti-spoofing', 'asvspoof', 'speech', 'deepfake-detection', 'en', 'dataset:asvspoof2019', 'license:mit', 'region:us']


In [26]:
from huggingface_hub import hf_hub_download

readme = hf_hub_download(
    repo_id="caa-speech-detection-asvspoof2019/rawnet2",
    filename="README.md"
)

print(open(readme, "r", encoding="utf-8").read())

README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

---
license: mit
language: en
library_name: pytorch
tags:
  - audio-classification
  - anti-spoofing
  - asvspoof
  - speech
  - deepfake-detection
datasets:
  - asvspoof2019
pipeline_tag: audio-classification
metrics:
  - eer
  - t-dcf
---

> **Improved version available:** [caa-speech-detection-asvspoof2019/rawnet2-plateau](https://huggingface.co/caa-speech-detection-asvspoof2019/rawnet2-plateau) — rawnet2_plateau (plateau LR scheduler) achieves **16.79% eval EER** (vs 15.09% here, note: this is a harder run; plateau scheduling significantly improves dev EER to 0.0090%) and **0.5065 tandem min t-DCF**.

# RawNet2 — AI-Generated Speech Detector

Binary bonafide-vs-spoof classifier operating directly on raw audio waveforms, trained on ASVspoof 2019 LA.

## Model Details

**Local version:** `rawnet2_weighted`

**Architecture:** RawNet2 (Tak et al., ICASSP 2021)

| Component | Config |
|-----------|--------|
| Sinc filterbank front-end | 128 filters, kernel length 129, linear frequency s

In [27]:
from huggingface_hub import model_info

info = model_info("SpeechAntiSpoofingBenchmarks/W2V2-AASIST")

print("Model:", info.id)
print("Downloads:", info.downloads)
print("Tags:", info.tags)

Model: SpeechAntiSpoofingBenchmarks/W2V2-AASIST
Downloads: 23
Tags: ['onnx', 'audio', 'anti-spoofing', 'audio-deepfake-detection', 'speech', 'asvspoof', 'wav2vec2', 'arxiv:2202.12233', 'license:mit', 'region:us']


In [28]:
from huggingface_hub import hf_hub_download

readme = hf_hub_download(
    repo_id="SpeechAntiSpoofingBenchmarks/W2V2-AASIST",
    filename="README.md"
)

print(open(readme, "r", encoding="utf-8").read())

README.md:   0%|          | 0.00/11.1k [00:00<?, ?B/s]

---
license: mit
tags:
  - audio
  - anti-spoofing
  - audio-deepfake-detection
  - speech
  - asvspoof
  - wav2vec2
---

# W2V2-AASIST

[![EER% 0.22 on ASVspoof2019_LA](https://img.shields.io/badge/EER%25%20on%20ASVspoof2019__LA-0.22%25-brightgreen)](https://huggingface.co/spaces/SpeechAntiSpoofingBenchmarks/SpeechAntiSpoofingArena?system=w2v2-aasist)
[![EER% 8.11 on ASVspoof2021_LA](https://img.shields.io/badge/EER%25%20on%20ASVspoof2021__LA-8.11%25-green)](https://huggingface.co/spaces/SpeechAntiSpoofingBenchmarks/SpeechAntiSpoofingArena?system=w2v2-aasist)
[![EER% 8.32 on ASVspoof2021_DF](https://img.shields.io/badge/EER%25%20on%20ASVspoof2021__DF-8.32%25-green)](https://huggingface.co/spaces/SpeechAntiSpoofingBenchmarks/SpeechAntiSpoofingArena?system=w2v2-aasist)
[![EER% 11.22 on InTheWild](https://img.shields.io/badge/EER%25%20on%20InTheWild-11.22%25-green)](https://huggingface.co/spaces/SpeechAntiSpoofingBenchmarks/SpeechAntiSpoofingArena?system=w2v2-aasist)
[![EER% 38.57 on CD-

In [29]:
from huggingface_hub import list_models

models = list(list_models(
    search="Nes2Net ASVspoof",
    limit=10
))

print("Models found:", len(models))

for m in models:
    print(m.id)

Models found: 0


In [30]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  43% /


In [31]:
from huggingface_hub import snapshot_download

repo_path = snapshot_download(
    repo_id="SpeechAntiSpoofingBenchmarks/W2V2-AASIST",
    allow_patterns=[
        "LA_model.pth",
        "w2v2_aasist.py",
        "_net.py"
    ]
)

print("Files downloaded!")
print(repo_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Files downloaded!
/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/snapshots/196128e5a5101d5cb6ac7701597891bc7de7e7b5


In [32]:
import os

repo_path = "/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/snapshots/196128e5a5101d5cb6ac7701597891bc7de7e7b5"

print(os.listdir(repo_path))

['LA_model.pth', 'README.md']


In [33]:
from huggingface_hub import hf_hub_download

repo_id = "SpeechAntiSpoofingBenchmarks/W2V2-AASIST"

file1 = hf_hub_download(
    repo_id=repo_id,
    filename="w2v2_aasist.py"
)

file2 = hf_hub_download(
    repo_id=repo_id,
    filename="_net.py"
)

print("Downloaded:")
print(file1)
print(file2)

RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-6a903ddf-22389fc7596a4dd73a9f4787;ac438f0e-89ff-4816-b421-e9054c44a7c5)

Entry Not Found for url: https://huggingface.co/SpeechAntiSpoofingBenchmarks/W2V2-AASIST/resolve/main/w2v2_aasist.py.

In [34]:
from huggingface_hub import list_repo_files

files = list_repo_files("SpeechAntiSpoofingBenchmarks/W2V2-AASIST")

for f in files:
    print(f)

.eval_results/SpeechAntiSpoofingBenchmarks/ADD2023_track12_test_r1/scores.txt
.eval_results/SpeechAntiSpoofingBenchmarks/ADD22_eval_31/scores.txt
.eval_results/SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA/result.yaml
.eval_results/SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA/scores.txt
.eval_results/SpeechAntiSpoofingBenchmarks/ASVspoof2021_DF/result.yaml
.eval_results/SpeechAntiSpoofingBenchmarks/ASVspoof2021_DF/scores.txt
.eval_results/SpeechAntiSpoofingBenchmarks/ASVspoof2021_LA/result.yaml
.eval_results/SpeechAntiSpoofingBenchmarks/ASVspoof2021_LA/scores.txt
.eval_results/SpeechAntiSpoofingBenchmarks/ASVspoof5/result.yaml
.eval_results/SpeechAntiSpoofingBenchmarks/ASVspoof5/scores.txt
.eval_results/SpeechAntiSpoofingBenchmarks/ArAD/result.yaml
.eval_results/SpeechAntiSpoofingBenchmarks/ArAD/scores.txt
.eval_results/SpeechAntiSpoofingBenchmarks/CD-ADD/result.yaml
.eval_results/SpeechAntiSpoofingBenchmarks/CD-ADD/scores.txt
.eval_results/SpeechAntiSpoofingBenchmarks/CFAD/result.yaml


In [35]:
from huggingface_hub import hf_hub_download

onnx_file = hf_hub_download(
    repo_id="SpeechAntiSpoofingBenchmarks/W2V2-AASIST",
    filename="w2v2-aasist.onnx"
)

print("ONNX model downloaded:")
print(onnx_file)

w2v2-aasist.onnx: reconstructing file:   0%|          |  0.00B / 1.26GB            

w2v2-aasist.onnx: downloading bytes:           |  0.00B            

ONNX model downloaded:
/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/snapshots/196128e5a5101d5cb6ac7701597891bc7de7e7b5/w2v2-aasist.onnx


In [36]:
!pip install -q onnxruntime

import onnxruntime as ort

onnx_path = "/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/snapshots/196128e5a5101d5cb6ac7701597891bc7de7e7b5/w2v2-aasist.onnx"

session = ort.InferenceSession(onnx_path)

print("INPUT:")
for x in session.get_inputs():
    print(x.name, x.shape, x.type)

print("\nOUTPUT:")
for x in session.get_outputs():
    print(x.name, x.shape, x.type)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 42.3 MB/s eta 0:00:00
INPUT:
wav ['batch', 64600] tensor(float)

OUTPUT:
logits ['batch', 2] tensor(float)


In [37]:
import numpy as np
import torch

# Get one sample
sample = next(iter(ds))

# Extract audio
audio_data = sample["audio"].get_all_samples()
waveform = audio_data.data
sample_rate = audio_data.sample_rate

# Convert to numpy
waveform = waveform.squeeze().numpy().astype(np.float32)

# Make sure it is 16 kHz
print("Sample rate:", sample_rate)
print("Original samples:", len(waveform))

# Model requires exactly 64600 samples
target_length = 64600

if len(waveform) >= target_length:
    waveform = waveform[:target_length]
else:
    # Repeat audio if shorter
    repeats = int(np.ceil(target_length / len(waveform)))
    waveform = np.tile(waveform, repeats)[:target_length]

# Add batch dimension
input_audio = waveform[np.newaxis, :].astype(np.float32)

print("Final input shape:", input_audio.shape)

# Run W2V2-AASIST
result = session.run(
    None,
    {"wav": input_audio}
)

logits = result[0]

# Convert logits to probabilities
probabilities = torch.softmax(
    torch.tensor(logits),
    dim=1
).numpy()[0]

# IMPORTANT:
# W2V2-AASIST: class 0 = spoof/fake
#              class 1 = bona fide/real

fake_probability = float(probabilities[0])
real_probability = float(probabilities[1])

if fake_probability >= real_probability:
    prediction = "AI_GENERATED"
else:
    prediction = "GENUINE"

print("\nPrediction:", prediction)
print("Real probability:", real_probability)
print("Fake probability:", fake_probability)
print("Actual dataset label:", sample["label"])

Sample rate: 16000
Original samples: 22801
Final input shape: (1, 64600)

Prediction: AI_GENERATED
Real probability: 3.0079334010224557e-06
Fake probability: 0.9999970197677612
Actual dataset label: 1


In [38]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

results = []

# 100 REAL + 100 FAKE
test_samples = real_samples + fake_samples

for i, sample in enumerate(test_samples):

    # Extract waveform
    audio_data = sample["audio"].get_all_samples()
    waveform = audio_data.data.squeeze().numpy().astype(np.float32)

    # Make exactly 64600 samples
    target_length = 64600

    if len(waveform) >= target_length:
        waveform = waveform[:target_length]
    else:
        repeats = int(np.ceil(target_length / len(waveform)))
        waveform = np.tile(waveform, repeats)[:target_length]

    # Batch dimension
    input_audio = waveform[np.newaxis, :]

    # Run model
    logits = session.run(
        None,
        {"wav": input_audio}
    )[0]

    # W2V2-AASIST:
    # class 0 = SPOOF
    # class 1 = BONAFIDE
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[0]

    fake_prob = float(probs[0])
    real_prob = float(probs[1])

    # Model prediction
    prediction = 1 if real_prob >= fake_prob else 0

    # Dataset:
    # 0 = REAL
    # 1 = FAKE
    actual = 1 if sample["label"] == 1 else 0

    results.append({
        "actual": actual,
        "prediction": prediction,
        "real_probability": real_prob,
        "fake_probability": fake_prob
    })

    if (i + 1) % 20 == 0:
        print(f"Processed {i+1}/200")

# Metrics
y_true = [x["actual"] for x in results]
y_pred = [x["prediction"] for x in results]

print("\n===== W2V2-AASIST RESULTS =====")

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_true, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Processed 20/200
Processed 40/200
Processed 60/200
Processed 80/200
Processed 100/200
Processed 120/200
Processed 140/200
Processed 160/200
Processed 180/200
Processed 200/200

===== W2V2-AASIST RESULTS =====
Accuracy : 0.18
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0

Confusion Matrix:
[[ 36  64]
 [100   0]]


In [40]:
print(open(
    "/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/snapshots/196128e5a5101d5cb6ac7701597891bc7de7e7b5/trt_w2v2_aasist.py"
).read())

FileNotFoundError: [Errno 2] No such file or directory: '/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/snapshots/196128e5a5101d5cb6ac7701597891bc7de7e7b5/trt_w2v2_aasist.py'

In [41]:
import os

matches = []

for root, dirs, files in os.walk("/root/.cache/huggingface/hub"):
    if "trt_w2v2_aasist.py" in files:
        matches.append(os.path.join(root, "trt_w2v2_aasist.py"))

print("Found:", matches)

Found: []


In [42]:
import os

for root, dirs, files in os.walk("/root/.cache/huggingface/hub"):
    for file in files:
        if "w2v2" in file.lower() or "aasist" in file.lower():
            print(os.path.join(root, file))

/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/snapshots/196128e5a5101d5cb6ac7701597891bc7de7e7b5/w2v2-aasist.onnx
/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/.no_exist/196128e5a5101d5cb6ac7701597891bc7de7e7b5/w2v2_aasist.py


In [43]:
!git clone -q https://github.com/TakHemlata/SSL_Anti-spoofing.git

import os

for root, dirs, files in os.walk("SSL_Anti-spoofing"):
    for file in files:
        if file in ["w2v2_aasist.py", "_net.py"]:
            print(os.path.join(root, file))

In [44]:
import os

for root, dirs, files in os.walk("SSL-Anti-spoofing"):
    for file in files:
        if file in ["w2v2_aasist.py", "_net.py"]:
            print(os.path.join(root, file))

In [45]:
!find SSL-Anti-spoofing -type f | head -50

find: ‘SSL-Anti-spoofing’: No such file or directory


In [46]:
!git clone https://github.com/TakHemlata/SSL_Anti-spoofing.git

fatal: destination path 'SSL_Anti-spoofing' already exists and is not an empty directory.


In [47]:
!pwd
!ls -la

/content
total 20
drwxr-xr-x 1 root root 4096 Aug 27 14:08 .
drwxr-xr-x 1 root root 4096 Aug 27 12:42 ..
drwxr-xr-x 4 root root 4096 Aug 24 13:27 .config
drwxr-xr-x 1 root root 4096 Aug 24 13:28 sample_data
drwxr-xr-x 9 root root 4096 Aug 27 14:08 SSL_Anti-spoofing


In [48]:
!find SSL_Anti-spoofing -type f | grep -E "w2v2_aasist|_net"

SSL_Anti-spoofing/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1/examples/translation_moe/translation_moe_src/mean_pool_gating_network.py


In [49]:
import onnxruntime as ort

session = ort.InferenceSession(
    "/root/.cache/huggingface/hub/models--SpeechAntiSpoofingBenchmarks--W2V2-AASIST/snapshots/196128e5a5101d5cb6ac7701597891bc7de7e7b5/w2v2-aasist.onnx"
)

print("Inputs:")
for x in session.get_inputs():
    print(x.name, x.shape, x.type)

print("\nOutputs:")
for x in session.get_outputs():
    print(x.name, x.shape, x.type)

print("\nProviders:")
print(session.get_providers())

Inputs:
wav ['batch', 64600] tensor(float)

Outputs:
logits ['batch', 2] tensor(float)

Providers:
['CPUExecutionProvider']


In [50]:
# Check model behaviour on one REAL and one FAKE sample

import numpy as np
import torch

def test_one(sample, name):

    audio_data = sample["audio"].get_all_samples()
    waveform = audio_data.data.squeeze().numpy().astype(np.float32)

    target_length = 64600

    if len(waveform) >= target_length:
        waveform = waveform[:target_length]
    else:
        repeats = int(np.ceil(target_length / len(waveform)))
        waveform = np.tile(waveform, repeats)[:target_length]

    waveform = waveform[np.newaxis, :].astype(np.float32)

    logits = session.run(
        None,
        {"wav": waveform}
    )[0][0]

    print("\n", name)
    print("Dataset label:", sample["label"])
    print("Raw logits:", logits)

    probs = torch.softmax(torch.tensor(logits), dim=0).numpy()

    print("Probability class 0 (SPOOF):", float(probs[0]))
    print("Probability class 1 (REAL): ", float(probs[1]))


test_one(real_samples[0], "KNOWN REAL SAMPLE")
test_one(fake_samples[0], "KNOWN FAKE SAMPLE")


 KNOWN REAL SAMPLE
Dataset label: 0
Raw logits: [ 6.7620163 -5.916769 ]
Probability class 0 (SPOOF): 0.9999969005584717
Probability class 1 (REAL):  3.116536163361161e-06

 KNOWN FAKE SAMPLE
Dataset label: 1
Raw logits: [ 7.1695666 -6.224147 ]
Probability class 0 (SPOOF): 0.9999984502792358
Probability class 1 (REAL):  1.5246974953697645e-06


In [51]:
from sklearn.metrics import roc_auc_score, roc_curve
import numpy as np

# For W2V2-AASIST:
# dataset label: 0 = REAL, 1 = FAKE
# model score: higher class-1 probability = more REAL

y_true = np.array([r["actual"] for r in results])

# REAL probability
real_scores = np.array([
    r["real_probability"] for r in results
])

# Convert so that HIGHER score = more likely FAKE
fake_scores = 1 - real_scores

fpr, tpr, thresholds = roc_curve(y_true, fake_scores)

fnr = 1 - tpr

eer_index = np.nanargmin(np.abs(fpr - fnr))

eer = (fpr[eer_index] + fnr[eer_index]) / 2

auc = roc_auc_score(y_true, fake_scores)

print("===== W2V2-AASIST EER/AUC =====")
print("EER :", eer)
print("EER %:", eer * 100)
print("AUC :", auc)

===== W2V2-AASIST EER/AUC =====
EER : 0.060000000000000026
EER %: 6.000000000000003
AUC : 0.9846999999999999
